In [5]:
import pandas as pd

# MLP

In [10]:
df1 = pd.read_csv("../data/poi_matches.csv")
counts = df1["poi_id"].value_counts()
valid_ids = counts[counts > 430].index
df1 = df1[df1["poi_id"].isin(valid_ids)]

df1['date'] = pd.to_datetime(df1['date'])
df1['month'] = df1['date'].dt.month
df1['day_name'] = df1['date'].dt.day_name()
df1['is_weekend'] = df1['date'].dt.dayofweek >= 5
df1['is_weekend'] = df1['is_weekend'].astype(int)

month_map = {
    1: "januari", 2: "februari", 3: "maart", 4: "april",
    5: "mei", 6: "juni", 7: "juli", 8: "augustus",
    9: "september", 10: "oktober", 11: "november", 12: "december"
}
df1['month'] = df1['month'].map(month_map)


df1['Seizoen'] = df1['month'].apply(lambda x: "winter" if x in [12, 1, 2] 
                                          else ("lente" if x in [3, 4, 5] 
                                          else ("zomer" if x in [6, 7, 8] 
                                          else "herfst")))


In [11]:
df2 = pd.read_csv("../data/pois.csv")

df2.rename(columns={'id': 'poi_id'}, inplace=True)
df2 = df2.drop(columns=['name'])
result = pd.merge(df1, df2, on='poi_id', how='inner')

result.to_csv("poi_matches_MLP_cleaned.csv", index=False)

In [ ]:
# features = [
#     'Duur_open_in_minuten', 'Startuur', 'Startminuut', 'Einduur', 'Eindminuut',
#     'Maand', 'Dag', 'Weekdag', 'Is_weekend',  
# ]

# RNN

In [ ]:
df = pd.read_csv("../data/poi_matches.csv")
counts = df["poi_id"].value_counts()
valid_ids = counts[counts > 430].index
df = df[df["poi_id"].isin(valid_ids)]

In [153]:
df.shape

(53894, 3)

In [154]:
df["date"] = pd.to_datetime(df["date"])
all_locations = []

for loc, group in df.groupby("poi_id"):
    group = group.sort_values("date")

    full_range = pd.date_range("2025-01-01", "2026-03-29", freq="D")

    group = group.set_index("date").reindex(full_range)

    group["poi_id"] = loc  # opnieuw invullen na reindex
    group.index.name = "date"
    group["matches"] = round(group["matches"].interpolate().bfill().ffill())

    all_locations.append(group)

df = pd.concat(all_locations).reset_index()